# Combine Extracted Kline CSVs By Time Range

This notebook uses **already extracted CSVs** (not ZIPs) and combines them by month range.

Workflow:
1. Read extracted monthly CSV files from a folder (for example `downloads/test1`).
2. Filter files by `START_YYYY_MM` to `END_YYYY_MM` (inclusive).
3. Merge them into one combined CSV.


In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import re

# -----------------------------
# Configuration
# -----------------------------
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "downloads").exists() and (PROJECT_ROOT.parent / "downloads").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SYMBOL = "BTCUSDT"
INTERVAL = "1s"
START_YYYY_MM = "2018-01"  # inclusive
END_YYYY_MM = "2020-02"    # inclusive

EXTRACTED_DIR = PROJECT_ROOT / "downloads" / "test1"
OUTPUT_DIR = EXTRACTED_DIR
COMBINED_OUTPUT_NAME = ""  # keep empty for auto name

KLINE_COLUMNS = ['Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close time', 'Quote asset volume', 'Number of trades', 'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore']

print(f"Project root : {PROJECT_ROOT}")
print(f"Extracted dir: {EXTRACTED_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Range        : {START_YYYY_MM} to {END_YYYY_MM}")


In [ ]:
def parse_yyyy_mm(text: str) -> datetime:
    return datetime.strptime(text, "%Y-%m")

start_dt = parse_yyyy_mm(START_YYYY_MM)
end_dt = parse_yyyy_mm(END_YYYY_MM)
if start_dt > end_dt:
    raise ValueError(f"START_YYYY_MM must be <= END_YYYY_MM ({START_YYYY_MM} > {END_YYYY_MM})")

if not EXTRACTED_DIR.exists():
    raise FileNotFoundError(f"Extracted directory not found: {EXTRACTED_DIR}")

csv_pattern = re.compile(
    rf"^{re.escape(SYMBOL)}-{re.escape(INTERVAL)}-(\d{{4}}-\d{{2}})(?:-part(\d+))?\.csv$",
    flags=re.IGNORECASE,
)

selected_csvs = []
for csv_path in EXTRACTED_DIR.iterdir():
    if not csv_path.is_file() or csv_path.suffix.lower() != ".csv":
        continue
    match = csv_pattern.match(csv_path.name)
    if not match:
        continue

    month_text = match.group(1)
    part_text = match.group(2)
    part_num = int(part_text) if part_text else 1
    month_dt = parse_yyyy_mm(month_text)

    if start_dt <= month_dt <= end_dt:
        selected_csvs.append((month_dt, part_num, month_text, csv_path))

selected_csvs.sort(key=lambda x: (x[0], x[1], x[3].name))

if not selected_csvs:
    raise ValueError(
        f"No extracted CSV files matched range {START_YYYY_MM} to {END_YYYY_MM} in {EXTRACTED_DIR}"
    )

print(f"Selected extracted CSV files: {len(selected_csvs)}")
for i, (_, part_num, month_text, csv_path) in enumerate(selected_csvs, start=1):
    if i <= 10 or i > len(selected_csvs) - 3:
        part_label = f" part{part_num}" if "-part" in csv_path.stem else ""
        print(f"  {month_text}{part_label}: {csv_path.name}")
    elif i == 11:
        print("  ...")


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

combined_output_path = OUTPUT_DIR / (
    COMBINED_OUTPUT_NAME
    if COMBINED_OUTPUT_NAME
    else f"{SYMBOL}-{INTERVAL}-{START_YYYY_MM}_to_{END_YYYY_MM}-combined.csv"
)

total_rows = 0
files_merged = 0

with combined_output_path.open("w", newline="", encoding="utf-8") as dst:
    writer = csv.writer(dst)
    writer.writerow(KLINE_COLUMNS)

    for _, _, _, csv_path in selected_csvs:
        with csv_path.open("r", newline="", encoding="utf-8") as src:
            reader = csv.reader(src)
            first_row = next(reader, None)
            if first_row is None:
                continue

            files_merged += 1

            if first_row != KLINE_COLUMNS:
                writer.writerow(first_row)
                total_rows += 1

            for row in reader:
                writer.writerow(row)
                total_rows += 1

print(f"Combined CSV: {combined_output_path}")
print(f"CSV files merged: {files_merged}")
print(f"Total data rows in combined CSV: {total_rows}")


In [ ]:
print("Preview (first 6 lines):")
with combined_output_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 5:
            break
